# 01 Single Logic Run

Run one symbol/timeframe with one fixed execution config. Use this notebook to inspect signal loading, next-open entry, SL/TP levels, cluster behavior, skip reasons, ambiguity, and cluster-level R metrics.

Mục đích notebook: chạy thử một bộ signal với một cấu hình SL/TP cố định, sau đó xem kết quả tổng quan và danh sách cluster để hiểu chiến lược đang hoạt động ra sao.

In [ ]:
# Cell 2 - Tìm project root và chuẩn bị các đường dẫn dùng chung.
# Cell này giúp notebook import được package backtest_optimize dù mở từ Jupyter ở thư mục khác.
from pathlib import Path
import sys

def find_project_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        Path("Z:/SEN05_Autotrading"),
        Path("//10.11.12.6/Share/SEN05_Autotrading"),
    ]
    for candidate in candidates:
        current = candidate
        while True:
            if (current / "pyproject.toml").exists() and (current / "backtest_optimize").exists():
                return current
            if current.parent == current:
                break
            current = current.parent
    raise RuntimeError("Could not find SEN05_Autotrading project root.")

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Các folder gốc được những cell sau dùng để đọc signal và lưu kết quả.
BACKTEST_ROOT = project_root / "backtest_optimize"
RAW_SIGNALS = project_root / "raw_signals"
OUTPUT_DIR = BACKTEST_ROOT / "outputs" / "single_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

project_root

In [ ]:
# Cell 3 - Import thư viện và module nội bộ cần cho single backtest.
# pandas dùng để xem bảng; các module backtest_optimize xử lý load signal, load giá, chạy engine, tính metric và lưu snapshot.
import importlib
import pandas as pd
from IPython.display import HTML, Markdown, display

from backtest_optimize.contracts import AmbiguityPolicy, MarketSpec
from backtest_optimize.io.signal_loader import load_signal_csv
from backtest_optimize.io.market_data import load_ohlcv_from_core
from backtest_optimize.execution.engine import run_single
from backtest_optimize.execution.sl_calculator import DEFAULT_COMBO_X_BUFFER
from backtest_optimize.execution.tp_calculator import DEFAULT_KTP_LEVEL, KTP_FIB_LEVELS
import backtest_optimize.analysis.notebook_dashboard as notebook_dashboard
from backtest_optimize.analysis.chart_payload import build_signal_chart_payload, render_signal_chart_html

notebook_dashboard = importlib.reload(notebook_dashboard)
from backtest_optimize.analysis.notebook_dashboard import (
    build_run_config,
    cluster_review_frame,
    cluster_review_html,
    control_panel_frame,
    dashboard_cards_html,
    dashboard_interpretation_frame,
    dashboard_metric_frame,
    discover_signal_catalog,
    run_context_frame,
    select_signal,
    status_breakdown_frame,
    style_dashboard_frame,
    style_report,
    warning_notes_frame,
)
from backtest_optimize.analysis.metrics import clusters_to_frame, summarize
from backtest_optimize.analysis.versioning import save_snapshot

# Tăng số cột/độ rộng hiển thị để bảng summary và clusters dễ đọc hơn trong notebook.
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

In [ ]:
# Cell 4 - Chọn bộ input và cấu hình backtest cho lần chạy này.
# Đây là cell chính bạn thường chỉnh trước khi bấm Run các cell phía sau.

# Signal controls. SIGNAL_CHOICE = "auto" sẽ chọn theo PREFERRED_SYMBOL/PREFERRED_TIMEFRAME.
SIGNAL_STRATEGY = "combo"
SIGNAL_CHOICE = "auto"
PREFERRED_SYMBOL = "US30"
PREFERRED_TIMEFRAME = "H4"

# Data controls.
WARMUP_BARS = 0

# Execution controls. Entry vẫn là next-open theo engine; các biến dưới chỉ điều khiển SL/TP/risk.
ACCOUNT_SIZE = 10_000.0
RISK_PER_CLUSTER = 0.01
X_BUFFER = DEFAULT_COMBO_X_BUFFER
TP_PROFILE = "combo_fib_atr"  # "combo_fib_atr" hoặc "risk_multiple"
KTP_LEVEL = DEFAULT_KTP_LEVEL
R_MULTIPLES = [1.0, 2.0, 3.0]
AMBIGUITY_POLICY = AmbiguityPolicy.CONSERVATIVE
SL_MOVE_RULE = "none"

# Market spec controls. Cần kiểm chứng lại trước khi tin sizing/cost tuyệt đối.
MARKET_SPEC = MarketSpec(
    symbol=PREFERRED_SYMBOL,
    pip_size=1.0,
    pip_value_per_lot=1.0,
    min_lot=0.01,
    lot_step=0.01,
    commission_per_lot_per_side=0.0,
    spread_buffer_pips=0.0,
    slippage_buffer_pips=0.0,
)

signal_catalog = discover_signal_catalog(RAW_SIGNALS, SIGNAL_STRATEGY)
selected_signal = select_signal(
    signal_catalog,
    choice=SIGNAL_CHOICE,
    symbol=PREFERRED_SYMBOL,
    timeframe=PREFERRED_TIMEFRAME,
)
SIGNAL_FILE = selected_signal.path
SYMBOL = selected_signal.symbol or PREFERRED_SYMBOL
TIMEFRAME = selected_signal.timeframe or PREFERRED_TIMEFRAME
MARKET_SPEC = MarketSpec(**{**MARKET_SPEC.__dict__, "symbol": SYMBOL})

RUN_CONFIG = build_run_config(
    account_size=ACCOUNT_SIZE,
    risk_per_cluster=RISK_PER_CLUSTER,
    x_buffer=X_BUFFER,
    tp_profile=TP_PROFILE,
    ktp_level=KTP_LEVEL,
    r_multiples=R_MULTIPLES,
    ambiguity_policy=AMBIGUITY_POLICY,
    sl_move_rule=SL_MOVE_RULE,
)

display(Markdown("### Control Panel"))
catalog_view = signal_catalog.drop(columns=["path"], errors="ignore").copy()
display(style_report(catalog_view))
display(style_report(control_panel_frame(
    selection=selected_signal,
    run_config=RUN_CONFIG,
    market_spec=MARKET_SPEC,
    warmup_bars=WARMUP_BARS,
)))

In [ ]:
# Cell 5 - Load signal và dữ liệu OHLCV tương ứng từ core data.
# Cell này kiểm tra được số lượng signal/bar và preview vài dòng đầu để xác nhận dữ liệu đọc đúng.
signals = load_signal_csv(SIGNAL_FILE, symbol=SYMBOL, timeframe=TIMEFRAME)

# Khoảng dữ liệu giá được lấy từ ngày signal đầu tiên đến sau signal cuối 10 ngày để đủ bar cho lệnh cuối thoát.
start = signals["bartime"].min()
end = signals["bartime"].max() + pd.Timedelta(days=10)
bars = load_ohlcv_from_core(SYMBOL, TIMEFRAME, start=start, end=end, warmup_bars=WARMUP_BARS, tail_bars=5)

display(Markdown("### Data Context"))
display(style_report(run_context_frame(
    signals=signals,
    bars=bars,
    selection=selected_signal,
    run_config=RUN_CONFIG,
)))

display(Markdown("### Data Preview"))
display(signals.head())
display(bars.head())

In [ ]:
# Cell 6 - Hiển thị chart giá và signal bằng Lightweight Charts.
# Cell này dùng bars/signals đã load ở Cell 5; không vẽ SL/TP để chỉ kiểm tra tín hiệu BUY/SELL trên giá.
CHART_MAX_BARS = 3000

chart_payload = build_signal_chart_payload(
    bars=bars,
    signals=signals,
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    max_bars=CHART_MAX_BARS,
)

display(HTML(render_signal_chart_html(chart_payload, height=760)))


In [ ]:
# Cell 7 - Chạy engine backtest và chuyển kết quả sang bảng dễ xem.
# Engine sẽ vào lệnh ở next-open, gom lệnh theo cluster, xử lý SL/TP/ambiguity và trả về R-metrics.
result = run_single(
    signals=signals,
    bars=bars,
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    market_spec=MARKET_SPEC,
    **RUN_CONFIG,
)

summary = summarize(result)
clusters = clusters_to_frame(result)

# Dashboard controls: đổi mode sang "worst", "best", "closed", "reversed", "skipped", hoặc "open" nếu muốn soi nhóm riêng.
CLUSTER_REVIEW_LIMIT = 80
CLUSTER_REVIEW_MODE = "first"

display(Markdown("## Backtest Dashboard"))
display(HTML(dashboard_cards_html(summary, RUN_CONFIG, MARKET_SPEC)))

display(Markdown("### Quick Reading"))
quick_reading = dashboard_interpretation_frame(summary, RUN_CONFIG, MARKET_SPEC)
display(style_dashboard_frame(quick_reading))

display(Markdown("### Metric Detail"))
metric_detail = dashboard_metric_frame(summary)
display(style_dashboard_frame(metric_detail))

display(Markdown("### Status Breakdown"))
status_breakdown = status_breakdown_frame(clusters)
display(style_dashboard_frame(status_breakdown))

display(Markdown("### Notes"))
notes_report = warning_notes_frame(summary, RUN_CONFIG, MARKET_SPEC)
display(style_dashboard_frame(notes_report))

display(Markdown(f"### Cluster Review - {CLUSTER_REVIEW_MODE} {CLUSTER_REVIEW_LIMIT}"))
cluster_review = cluster_review_frame(clusters, limit=CLUSTER_REVIEW_LIMIT, mode=CLUSTER_REVIEW_MODE)
display(HTML(cluster_review_html(cluster_review)))

In [ ]:
# Cell 8 - Lưu kết quả ra file để audit, so sánh version hoặc dùng lại ở notebook khác.
# Mỗi lần chạy tạo một run_name theo symbol/timeframe/timestamp để không ghi đè kết quả cũ.
run_name = f"single_{SYMBOL}_{TIMEFRAME}_{pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M%S')}"
clusters_path = OUTPUT_DIR / f"{run_name}_clusters.csv"
summary_path = OUTPUT_DIR / f"{run_name}_summary.csv"

clusters.to_csv(clusters_path, index=False)
pd.DataFrame([summary]).to_csv(summary_path, index=False)

# Snapshot lưu kèm config, assumptions và thông tin repo để truy vết lần chạy.
snapshot_path = save_snapshot(
    name=run_name,
    config={**RUN_CONFIG, "market_spec": MARKET_SPEC, "symbol": SYMBOL, "timeframe": TIMEFRAME},
    result_summary=summary,
    signal_file=SIGNAL_FILE,
    market_data_source_id="core_python.data.loader",
    assumptions=result.assumptions,
    repo_root=project_root,
)

print(clusters_path)
print(summary_path)
print(snapshot_path)